### Create catalog and schemas

In [0]:
%sql
create schema if not exists olist_ecommerce.bronze;
create schema if not exists olist_ecommerce.silver;
create schema if not exists olist_ecommerce.gold;

### Create Bronze control table

In [0]:
spark.sql('''
    create table if not exists olist_ecommerce.bronze.ingestion_control(
        layer String,
        table_name String,
        source_path String,
        source_format String,
        last_run_id String,
        rows_written Bigint,
        run_status STring,
        bronze_ingested_at timestamp,
        updated_at timestamp
    )
    using delta
          ''')

### Adls path

In [0]:
bronze_base_path = "abfss://bronze@olistecommercesc.dfs.core.windows.net/"

### Create table configuration

In [0]:
tables_config = {
    "customers":{
        "path": bronze_base_path + "customers.csv",
        "format": "csv",
        "source_system": "onprem"
    },
     "sellers":{
        "path": bronze_base_path + "sellers.csv",
        "format": "csv",
        "source_system": "onprem"
    }, 
     "orders":{
        "path": bronze_base_path + "orders.csv",
        "format": "csv",
        "source_system": "azure_sql_db"
    }, 
     "order_items":{
        "path": bronze_base_path + "order_items.csv",
        "format": "csv",
        "source_system": "azure_sql_db"
    }, 
     "order_payments":{
        "path": bronze_base_path + "order_payments.csv",
        "format": "csv",
        "source_system": "azure_sql_db"
    }, 
     "geolocation":{
        "path": bronze_base_path + "geolocation.csv",
        "format": "csv",
        "source_system": "adls_raw"
    }, 
     "order_reviews":{
        "path": bronze_base_path + "order_reviews.csv",
        "format": "csv",
        "source_system": "adls_raw"
    }, 
     "products":{
        "path": bronze_base_path + "products.csv",
        "format": "csv",
        "source_system": "github"
    },
     "product_category_name_translation":{
        "path": bronze_base_path + "product_category_name_translation.csv",
        "format": "csv",
        "source_system": "github"
    }
}

### Create Bronze load function

In [0]:
from pyspark.sql.functions import *
from datetime import datetime
import uuid

bronze_run_id = str(uuid.uuid4())

def read_source_file(path, file_format):
    if file_format == "csv" :
        return (
            spark.read.option("header", True)
            .option("inferSchema", True)
            .csv(path)
        )
    else:
        raise ValueError(f"Unsupported format: {file_format}")


### Load adls files into bronze dalta tables

In [0]:
for table_name, cfg in tables_config.items():
    print(f"Processing {table_name}")

    df = read_source_file(cfg["path"], cfg["format"])

    df = (
        df.withColumn("bronze_ingested_at", current_timestamp())
        .withColumn("bronze_run_id", lit(bronze_run_id))
        .withColumn("source_system", lit(cfg["source_system"]))
        .withColumn("source_file", col("_metadata.file_path"))
    )

    row_count = df.count()

    target_table = f"olist_ecommerce.bronze.{table_name}_raw"

    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(target_table)
    )

    control_df = spark.createDataFrame(
        [(
            "bronze",
            table_name,
            cfg["path"],
            cfg["format"],
            bronze_run_id,
            row_count,
            "success",
            datetime.utcnow(),
            datetime.utcnow()
        )],
        '''
        layer String,
        table_name String,
        source_path String,
        source_format String,
        last_run_id String,
        rows_written Bigint,
        run_status STring,
        bronze_ingested_at timestamp,   
        updated_at timestamp
        '''
    )

    (
        control_df.write.format("delta")
        .mode("append")
        .saveAsTable("olist_ecommerce.bronze.ingestion_control")
    )

    print(f"Loaded {row_count} rows into {target_table}")

In [0]:
display (spark.sql('''
          select * from olist_ecommerce.bronze.ingestion_control
          order by updated_at desc
          '''
) )

In [0]:
for table_name in tables_config.keys():
    count = spark.table(f"olist_ecommerce.bronze.{table_name}_raw").count()
    print(table_name, count)

In [0]:
tables_config["products"]